# 🏇 KEIBA-AI 日曜結果 v5

**日曜18時以降に実行**
- 日曜レース結果スクレイピング
- 投資対照収益率集計
- アプリJSONに保存

| ステップ | 説明 |
|---|---|
| 1 | セットアップ |
| 2 | 定義 |
| 3 | ✅ 結果スクレイピング |


In [ ]:
!pip install -q requests beautifulsoup4 lxml pandas
from google.colab import drive, userdata
drive.mount('/content/drive')
import os, json, re, time, sqlite3, requests, subprocess
import pandas as pd
from bs4 import BeautifulSoup
from datetime import datetime, timezone, timedelta

BASE_DIR = '/content/drive/MyDrive/keiba_ai'
DB_PATH  = f'{BASE_DIR}/data/keiba.db'

def get_jst_now():
    try:
        subprocess.run(['sudo','ntpdate','-u','ntp.nict.jp'],
                       capture_output=True, text=True, timeout=10)
    except: pass
    return datetime.now(timezone(timedelta(hours=9)))

print(f'✅ セットアップ完了: {get_jst_now().strftime("%Y-%m-%d %H:%M:%S")}')


## セル3b: src/ モジュールセットアップ（初回のみ）


In [ ]:
# ── src/ モジュールをGoogle Driveにセットアップ ──────────────────
import os, subprocess

src_dst = f'{BASE_DIR}/src'
if not os.path.exists(src_dst):
    print('📦 GitHubからsrc/を取得中...')
    subprocess.run([
        'git', 'clone', '--depth=1', '--filter=blob:none', '--sparse',
        'https://github.com/hanagenuku/keiba_ai.git',
        '/tmp/keiba_ai_src'
    ], check=True, capture_output=True)
    subprocess.run([
        'git', '-C', '/tmp/keiba_ai_src', 'sparse-checkout', 'set', 'src'
    ], check=True, capture_output=True)
    import shutil
    shutil.copytree('/tmp/keiba_ai_src/src', src_dst)
    print(f'✅ src/ をセットアップしました: {src_dst}')
else:
    print(f'✅ src/ は既に存在します: {src_dst}')


In [ ]:
import sys
sys.path.insert(0, f'{BASE_DIR}/src')
try:
    from utils.config import PLACE_NAMES, PLACE_ENG, HEADERS, JRA_BASE
    print('✅ src/ モジュール読み込み成功')
except ImportError as e:
    print(f'⚠ src/ なし → ノートブック内定義を使用 ({e})')

PLACE_NAMES = {'01':'札幌','02':'函館','03':'福島','04':'新潟','05':'東京',
               '06':'中山','07':'中京','08':'京都','09':'阪神','10':'小倉'}
PLACE_ENG   = {'nakayama':'06','hanshin':'09','chukyo':'07','tokyo':'05',
               'kyoto':'08','sapporo':'01','hakodate':'02','fukushima':'03',
               'niigata':'04','kokura':'10'}
HEADERS  = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36'}
JRA_BASE = 'https://www.jra.go.jp'

def calc_suffix(r01, r):
    if r<=9:    return f'{(r01+(r-1)*181)%256:02X}'
    elif r==10: return f'{(r01+8*181+245)%256:02X}'
    else:       return f'{(r01+8*181+245+(r-10)*181)%256:02X}'

def find_r01_result(base, date, sess):
    for s in range(256):
        cn = f'{base}01{date}/{s:02X}'
        r  = sess.post(f'{JRA_BASE}/JRADB/accessS.html',
                       data={'CNAME':cn}, timeout=10)
        r.encoding = 'shift_jis'
        if 'パラメータエラー' not in r.text and BeautifulSoup(r.text,'lxml').find_all('table'):
            return s
        time.sleep(0.05)
    return None

def parse_dividends(soup):
    full = soup.get_text(' ', strip=True)
    divs = {}
    m = re.search(r'単勝\s+(\d+)\s+([\d,]+)\s+円', full)
    if m: divs['tansho'] = {'num':int(m.group(1)), 'payout':int(m.group(2).replace(',',''))}
    idx = full.find('複勝')
    if idx >= 0:
        fm = re.findall(r'(\d+)\s+([\d,]+)\s*円', full[idx:idx+300])
        if fm: divs['fukusho'] = [{'num':int(f[0]),'payout':int(f[1].replace(',',''))} for f in fm[:3]]
    idx = full.find('ワイド')
    if idx >= 0:
        wm = re.findall(r'(\d+)-(\d+)\s+([\d,]+)\s*円', full[idx:idx+300])
        if wm: divs['wide'] = [{'nums':[int(w[0]),int(w[1])],'payout':int(w[2].replace(',',''))} for w in wm[:3]]
    return divs

def fetch_sunday_results(sess):
    today = get_jst_now().strftime('%Y%m%d')
    print(f'📋 日曜レース結果取得中 ({today})...')
    resp = sess.post(f'{JRA_BASE}/JRADB/accessS.html', data={'CNAME':'pw01sli00/AF'}, timeout=15)
    resp.encoding = 'shift_jis'
    soup = BeautifulSoup(resp.text, 'lxml')
    kaisai = {}
    for tag in soup.find_all(onclick=True):
        oc = tag['onclick']
        m  = re.search(r'pw01srl1(\d{2})(\d{4})(\d{2})(\d{2})\d{2}(\d{8})/(\w{2})', oc)
        if m and m.group(5) == today:
            place, year, month, kai, date, r11_sx = m.groups()
            sde_base = f'pw01sde10{place}{year}{month}{kai}'
            kaisai[place] = {'sde_base':sde_base, 'date':date, 'r11_sx':r11_sx}
    try:
        resp2 = sess.get(f'{JRA_BASE}/keiba/rpdf/', timeout=15)
        resp2.encoding = 'shift_jis'
        for a in BeautifulSoup(resp2.text, 'lxml').find_all('a', href=True):
            href = a['href']
            if today not in href or '.pdf' not in href: continue
            m = re.search(r'(\d{8})-(\d{2})([a-z]+)(\d{2})\.pdf', href)
            if not m: continue
            date, seq, peng, kai = m.groups()
            pc = PLACE_ENG.get(peng)
            if not pc or pc in kaisai: continue
            year = date[:4]
            for mon in ['01','02','03','04']:
                sde_base = f'pw01sde10{pc}{year}{mon}{kai}'
                r01 = find_r01_result(sde_base, date, sess)
                if r01 is not None:
                    kaisai[pc] = {'sde_base':sde_base, 'date':date,
                                  'r11_sx':calc_suffix(r01,11), 'r01':r01}
                    print(f'  rpdf補完: {PLACE_NAMES.get(pc,"?")}')
                    break
    except Exception as e:
        print(f'  ⚠ rpdf補完失敗: {e}')
    all_results = []
    for pc, info in kaisai.items():
        sde_base = info['sde_base']; date = info['date']; rc = PLACE_NAMES.get(pc, '?')
        if 'r01' in info:
            r01_val = info['r01']
        else:
            r11_val = int(info['r11_sx'], 16)
            r01_val = (r11_val - 8*181 - 245) % 256
            if calc_suffix(r01_val, 11).upper() != info['r11_sx'].upper():
                r01_val = find_r01_result(sde_base, date, sess)
                if r01_val is None: continue
        print(f'\n🏟 {rc}')
        for r in range(1, 13):
            sx   = calc_suffix(r01_val, r)
            cn   = f'{sde_base}{r:02d}{date}/{sx}'
            resp = sess.post(f'{JRA_BASE}/JRADB/accessS.html', data={'CNAME':cn}, timeout=15)
            resp.encoding = 'shift_jis'
            if 'パラメータエラー' in resp.text: continue
            soup = BeautifulSoup(resp.text, 'lxml')
            if not soup.find_all('table'): continue
            tables = soup.find_all('table')
            header = tables[0].get_text(' ', strip=True)
            c2  = header.replace('国際競走','').replace('混合競走','')
            sp  = re.search(r'([\u3040-\u9fff\u30A0-\u30FFa-zA-Z0-9]+(?:賞|特別|記念|大賞|ステークス|カップ|グランプリ|トロフィ))', c2)
            gen = re.search(r'(\d歳(?:以上)?(?:未満)?(?:以上|未出走|1勝クラス|2勝クラス|3勝クラス|オープン))', header)
            rname = (sp.group(1).strip() if sp and sp.group(1) not in ('国際競走','混合競走') and len(sp.group(1))>=3
                     else gen.group(1).strip() if gen else f'R{r:02d}')
            dm   = re.search(r'([\d,]+)\s*メートル\s*[（(]([芝ダ])', header)
            dist = int(dm.group(1).replace(',','')) if dm else 2000
            surf = '芝' if dm and dm.group(2)=='芝' else 'ダート'
            race_id = f'{date}_{pc}_{r:02d}'
            finishers = []
            for row in tables[0].find_all('tr'):
                cells2 = row.find_all('td')
                if len(cells2) < 10: continue
                tx = [c.get_text(' ', strip=True) for c in cells2]
                pm = re.match(r'^(\d+)$', tx[0].strip())
                if not pm: continue
                place = int(pm.group(1))
                if place > 8: continue
                nm  = re.match(r'^(\d+)$', tx[2].strip())
                num = int(nm.group(1)) if nm else 0
                name_m = re.match(
                    r'^([\u30A0-\u30FF\u4e00-\u9fffA-Za-z][\u30A0-\u30FF\u4e00-\u9fffA-Za-z0-9ー]{1,20})',
                    tx[3].strip())
                name = name_m.group(1).strip() if name_m else tx[3].strip()[:10]
                pos_nums = re.findall(r'\d+', tx[9] if len(tx)>9 else '')
                if pos_nums:
                    positions = [int(n) for n in pos_nums[:4]]
                    first = positions[0]; avg = sum(positions)/len(positions)
                    style = '逃げ' if first==1 else '先行' if avg<=3 else '差し' if avg<=7 else '追込'
                else:
                    style = '差し'
                agari_m = re.search(r'(\d{2}\.\d)', tx[10]) if len(tx)>10 else None
                agari   = float(agari_m.group(1)) if agari_m else 0.0
                finishers.append({'place':place, 'num':num, 'name':name,
                                  'running_style':style, 'agari3f':agari})
            if not finishers: continue
            divs = parse_dividends(soup)
            all_results.append({'race_id':race_id, 'racecourse':rc, 'race_num':r,
                                 'race_name':rname, 'distance':dist, 'surface':surf,
                                 'finishers':finishers, 'dividends':divs})
            top3 = ' '.join(f"{h['place']}着#{h['num']}{h['name'][:4]}({h['running_style']})" for h in finishers[:3])
            print(f'  R{r:02d} {rname[:8]}: {top3}')
            time.sleep(0.8)
    print(f'\n📋 結果スクレイピング完了: {len(all_results)}レース')
    return all_results

print('✅ 日曜結果v5 定義完了')


In [ ]:
sess = requests.Session()
sess.headers.update(HEADERS)
sess.get(f'{JRA_BASE}/keiba/thisweek/', timeout=15)

jst_now2   = get_jst_now()
today_date = jst_now2.strftime('%Y-%m-%d')
print(f'現在時刻: {jst_now2.strftime("%Y-%m-%d %H:%M")}')

# ── 結果スクレイピング ────────────────────────────────────────────────
sun_results = fetch_sunday_results(sess)

# ── race_results テーブルに保存 ──────────────────────────────────────
if sun_results:
    conn = sqlite3.connect(DB_PATH)
    conn.execute('''CREATE TABLE IF NOT EXISTS race_results (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        race_id TEXT, racecourse TEXT, race_num INTEGER, race_name TEXT,
        distance INTEGER, surface TEXT, place INTEGER, horse_num INTEGER,
        horse_name TEXT, running_style TEXT, agari3f REAL,
        tansho_payout INTEGER, fukusho_payout INTEGER)''')
    for result in sun_results:
        divs     = result.get('dividends', {})
        tansho_n = divs.get('tansho', {}).get('num', 0)
        tansho_p = divs.get('tansho', {}).get('payout', 0)
        for h in result['finishers']:
            fuku = next((f['payout'] for f in divs.get('fukusho',[]) if f['num']==h['num']), 0)
            conn.execute(
                'INSERT OR IGNORE INTO race_results '
                '(race_id,racecourse,race_num,race_name,distance,surface,'
                'place,horse_num,horse_name,running_style,agari3f,tansho_payout,fukusho_payout) '
                'VALUES (?,?,?,?,?,?,?,?,?,?,?,?,?)',
                (result['race_id'], result['racecourse'], result['race_num'],
                 result['race_name'], result['distance'], result['surface'],
                 h['place'], h['num'], h['name'], h['running_style'], h['agari3f'],
                 tansho_p if h['num']==tansho_n else 0, fuku))
    conn.commit(); conn.close()
    print(f'✅ {len(sun_results)}レース保存')

# ── horse_history に保存（重複チェック付き） ─────────────────────────
hist_conn = sqlite3.connect(f'{BASE_DIR}/data/history.db')
saved_h = 0
for result in sun_results:
    race_id = result['race_id']
    exists = hist_conn.execute(
        'SELECT COUNT(*) FROM race_history WHERE race_id=?', (race_id,)).fetchone()[0]
    if not exists:
        hist_conn.execute(
            'INSERT OR IGNORE INTO race_history (race_id,date,racecourse,race_num,race_name,distance,surface) VALUES (?,?,?,?,?,?,?)',
            (race_id, result.get('date','20260426'), result['racecourse'], result['race_num'],
             result['race_name'], result['distance'], result['surface']))
    divs = result.get('dividends', {})
    tan_n = divs.get('tansho', {}).get('num', 0)
    tan_p = divs.get('tansho', {}).get('payout', 0)
    fuku_list = divs.get('fukusho', [])
    for h in result['finishers']:
        ex = hist_conn.execute(
            'SELECT COUNT(*) FROM horse_history WHERE race_id=? AND horse_num==?',
            (race_id, h['num'])).fetchone()[0]
        if ex: continue
        fuku_p = next((f['payout'] for f in fuku_list if f['num']==h['num']), 0)
        hist_conn.execute(
            'INSERT INTO horse_history (race_id,date,racecourse,race_name,distance,surface,'
            'place,horse_num,horse_name,running_style,agari3f,popularity,'
            'jockey,trainer,tansho_payout,fukusho_payout) VALUES (?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?)',
            (race_id, result.get('date','20260426'), result['racecourse'], result['race_name'],
             result['distance'], result['surface'],
             h['place'], h['num'], h['name'], h.get('running_style',''),
             h.get('agari3f', 0), h.get('popularity', 99),
             h.get('jockey',''), h.get('trainer',''),
             tan_p if h['num']==tan_n else 0, fuku_p))
        saved_h += 1
hist_conn.commit(); hist_conn.close()
print(f'✅ horse_history: {saved_h}件保存')

# ── 投資対照収益計算 ──────────────────────────────────────────────────
conn = sqlite3.connect(DB_PATH)
try:
    bets_df     = pd.read_sql('SELECT * FROM bets WHERE date=?', conn, params=[today_date])
    bet_records = bets_df.to_dict('records')
except Exception as e:
    print(f'⚠ 本日の賭け履歴: {e}')
    bet_records = []
conn.close()
print(f'賭け履歴件数: {len(bet_records)}件')

results_by_id = {r['race_id']: r for r in sun_results}
hit=0; total=0; invested=0; recovered=0; details=[]
conn = sqlite3.connect(DB_PATH)

for bet in bet_records:
    race_id = bet.get('race_id', '')
    result  = results_by_id.get(race_id)
    if not result: continue
    top3   = [h['num'] for h in result['finishers'][:3]]
    top1_n = top3[0] if top3 else 0
    divs   = result.get('dividends', {})
    total    += 1
    invested += int(bet['amount'])
    is_hit = False; payout = 0
    if bet['bet_type'] == '複勝' and int(bet['horse_num']) in top3:
        fuku   = next((f['payout'] for f in divs.get('fukusho',[]) if f['num']==int(bet['horse_num'])), 0)
        payout = int(bet['amount'] * fuku / 100); is_hit = True
    elif bet['bet_type'] == '単勝' and int(bet['horse_num']) == top1_n:
        payout = int(bet['amount'] * divs.get('tansho',{}).get('payout',0) / 100); is_hit = True
    elif bet['bet_type'] == 'ワイド':
        h1 = int(bet['horse_num']); h2 = int(bet.get('horse_num2', 0))
        for w in divs.get('wide', []):
            if h1 in w['nums'] and h2 in w['nums']:
                payout = int(bet['amount'] * w['payout'] / 100); is_hit = True; break
    recovered += payout
    if is_hit: hit += 1
    if int(bet.get('is_hit', -1)) == -1:
        conn.execute('UPDATE bets SET is_hit=?,payout=? WHERE id=?',
                     (1 if is_hit else 0, payout, bet['id']))
    rc   = result['racecourse']; rn = result['race_num']
    mark = '✅' if is_hit else '❌'
    details.append(
        f'  {mark} {rc}R{rn:02d} {bet["bet_type"]}#{int(bet["horse_num"])} ¥{int(bet["amount"]):,}'
        + (f' → ¥{payout:,}' if is_hit else ' → 外れ'))

conn.commit(); conn.close()

roi = recovered / invested * 100 if invested > 0 else 0
print(f'\n【日曜投資対照収益】')
for d in details: print(d)
print(f'\n的中率: {hit}/{total}件  ROI: {roi:.1f}%')
print(f'投資額: ¥{invested:,} / 回収額: ¥{recovered:,}')

# ── アプリJSONに結果を保存 ────────────────────────────────────────────
app_json_path = f'{BASE_DIR}/app/prediction_latest.json'
if os.path.exists(app_json_path):
    with open(app_json_path) as f: app_data = json.load(f)
else: app_data = {}
app_data['result'] = {
    'updated_at': jst_now2.isoformat(),
    'hit':hit, 'total':total, 'invested':invested,
    'recovered':recovered, 'roi':round(roi,1), 'details':details
}
with open(app_json_path, 'w', encoding='utf-8') as f:
    json.dump(app_data, f, ensure_ascii=False, indent=2)
print(f'\n✅ 結果をアプリJSONに保存')

log2 = f'{BASE_DIR}/logs/sunday_result_{jst_now2.strftime("%Y%m%d_%H%M")}.txt'
with open(log2, 'w') as f:
    f.write('\n'.join(
        [f'日曜結果 {jst_now2.strftime("%m/%d")}'] + details +
        [f'的中率:{hit}/{total}件 ROI:{roi:.1f}%', f'投資額:¥{invested:,} / 回収額:¥{recovered:,}']))
print(f'✅ ログ保存: {log2}')


In [ ]:
# =====================================================================
# 進捗ファイル更新（日曜結果v5）
# =====================================================================
progress_path = f'{BASE_DIR}/data/progress.md'
summary = f"""
## 最新結果（{jst_now2.strftime('%Y-%m-%d')}）
- 的中率: {hit}/{total}件
- 投資額: ¥{invested:,}
- 回収額: ¥{recovered:,}
- ROI: {roi:.1f}%
"""
if os.path.exists(progress_path):
    with open(progress_path, 'a', encoding='utf-8') as f:
        f.write(summary)
print('✅ 最新結果を進捗ファイルに追記完了')


In [ ]:
# =====================================================================
# results.csv に最新結果を保存
# =====================================================================
import pandas as pd
import json, os

res_path  = f'{BASE_DIR}/logs/results.csv'
pred_path = f'{BASE_DIR}/logs/predictions.csv'
meta_path = f'{BASE_DIR}/models/meta.json'

model_version = 'unknown'
if os.path.exists(meta_path):
    with open(meta_path) as f:
        model_version = json.load(f).get('current_model', 'unknown')

# 賭け結果をresults.csvに保存
conn2 = sqlite3.connect(DB_PATH)
bets_today = conn2.execute('''
    SELECT date, race_id, bet_type, horse_num, horse_name,
           odds_est, amount, is_hit, payout
    FROM bets
    WHERE date = ?
''', (today_date,)).fetchall()
conn2.close()

rows = []
for b in bets_today:
    date, race_id, bet_type, horse_num, horse_name, odds_est, amount, is_hit, payout = b
    rows.append({
        'date':          date,
        'race_id':       race_id,
        'horse_num':     horse_num,
        'horse_name':    horse_name,
        'bet_type':      bet_type,
        'odds':          odds_est,
        'amount':        amount,
        'is_hit':        is_hit,
        'payout':        payout if payout else 0,
        'model_version': model_version,
    })

if rows:
    df_res = pd.DataFrame(rows)
    df_res.to_csv(res_path, mode='a', header=False, index=False)
    print(f'✅ results.csv に{len(rows)}件保存')

# 通算ROI集計
if os.path.exists(res_path):
    df_all = pd.read_csv(res_path, names=[
        'date','race_id','horse_num','horse_name',
        'bet_type','odds_est','amount','is_hit','payout','model_version'
    ])
    total_inv_all = pd.to_numeric(df_all['amount'], errors='coerce').sum()
    total_ret_all = pd.to_numeric(df_all['payout'], errors='coerce').sum()

    roi_all = total_ret_all / total_inv_all * 100 if total_inv_all > 0 else 0
    print(f'\n【通算サマリ】')
    print(f'  通算投資額: ¥{total_inv_all:,.0f}')
    print(f'  通算回収額: ¥{total_ret_all:,.0f}')
    print(f'  通算ROI: {roi_all:.1f}%')
    print(f'  通算ベット数: {len(df_all)}件')


In [ ]:
# =====================================================================
# 📊 多軸ROI分析（EV順位・賭け種別・競馬場別・距離別・人気別・脚質別）
# =====================================================================
import sqlite3, pandas as pd

conn = sqlite3.connect(DB_PATH)
try:
    df_bets = pd.read_sql("""
        SELECT date, race_id, bet_type, horse_num, horse_name,
               odds_est, amount, is_hit, payout,
               ev_rank, racecourse, distance, surface,
               running_style, popularity, ai_score
        FROM bets
        WHERE is_hit >= 0
    """, conn)
finally:
    conn.close()

if df_bets.empty:
    print('⚠ 集計対象のベットデータがありません')
else:
    df_bets['amount']  = pd.to_numeric(df_bets['amount'],  errors='coerce').fillna(0)
    df_bets['payout']  = pd.to_numeric(df_bets['payout'],  errors='coerce').fillna(0)
    df_bets['ev_rank'] = pd.to_numeric(df_bets['ev_rank'], errors='coerce').fillna(99)
    df_bets['popularity'] = pd.to_numeric(df_bets['popularity'], errors='coerce').fillna(99)

    def roi_summary(g):
        inv = g['amount'].sum(); ret = g['payout'].sum()
        hit = g['is_hit'].sum(); n   = len(g)
        roi = ret / inv * 100 if inv > 0 else 0
        return pd.Series({'件数':n,'的中':int(hit),'的中率':f'{hit/n*100:.1f}%',
                          '投資額':int(inv),'回収額':int(ret),'ROI':f'{roi:.1f}%'})

    print('='*60)
    print('【① EV順位別 ROI】（EV1位が最も期待値が高い）')
    df_bets['EV順位'] = df_bets['ev_rank'].apply(
        lambda x: '1位' if x==1 else '2位' if x==2 else '3位' if x==3 else '4位以下')
    print(df_bets.groupby('EV順位').apply(roi_summary, include_groups=False).to_string())

    print()
    print('='*60)
    print('【② 賭け種別 ROI】')
    print(df_bets.groupby('bet_type').apply(roi_summary, include_groups=False).to_string())

    print()
    print('='*60)
    print('【③ 競馬場別 ROI】')
    if df_bets['racecourse'].str.strip().any():
        print(df_bets.groupby('racecourse').apply(roi_summary, include_groups=False).to_string())
    else:
        print('（データなし ─ 競馬場から再取得必要）')

    print()
    print('='*60)
    print('【④ 距離帯別 ROI】')
    df_bets['距離帯'] = pd.cut(
        pd.to_numeric(df_bets['distance'], errors='coerce').fillna(0),
        bins=[0,1400,1800,2200,9999],
        labels=['~1400m','1401~1800m','1801~2200m','2201m~'])
    print(df_bets.groupby('距離帯', observed=True).apply(roi_summary, include_groups=False).to_string())

    print()
    print('='*60)
    print('【⑤ 人気別 ROI】')
    df_bets['人気帯'] = df_bets['popularity'].apply(
        lambda x: '1~3番人気' if x<=3 else '4~6番人気' if x<=6 else '7~9番人気' if x<=9 else '10番人気以上')
    print(df_bets.groupby('人気帯').apply(roi_summary, include_groups=False).to_string())

    print()
    print('='*60)
    print('【⑥ 脚質別 ROI】')
    if df_bets['running_style'].str.strip().any():
        print(df_bets.groupby('running_style').apply(roi_summary, include_groups=False).to_string())
    else:
        print('（データなし ─ 脚質から再取得必要）')

    print()
    print('='*60)
    print('【⑥ 週次ROI推移】')
    df_bets['week'] = pd.to_datetime(df_bets['date']).dt.to_period('W').astype(str)
    wk = df_bets.groupby('week').apply(roi_summary, include_groups=False).reset_index()
    print(wk[['week','件数','的中率','ROI']].to_string(index=False))

    total_inv = df_bets['amount'].sum(); total_ret = df_bets['payout'].sum()
    total_roi = total_ret / total_inv * 100 if total_inv > 0 else 0
    print(f'\n📊 【通算サマリ】')
    print(f'  通算投資額: ¥{int(total_inv):,}  通算回収額: ¥{int(total_ret):,}  通算ROI: {total_roi:.1f}%  ベット数: {len(df_bets)}件')
